# 🎵 Brazilian Lyrics Analysis Pipeline - Regeneration Notebook

Este notebook regenera todos os arquivos processados do projeto de análise de letras brasileiras.

## O que este notebook faz:

1. ✅ **Setup do ambiente** - Clona o repositório e instala dependências
2. ✅ **Processa corpus** - Lê arquivos TXT de letras e gera corpus estruturado
3. ✅ **Gera análises** - Executa análises de padrões, inovação e distinctiveness
4. ✅ **Cria índices** - Gera índice do corpus e arquivos de referência

## Arquivos regenerados:

**Data Processada:**
- `corpus_index.json` - Índice de todas as músicas (título, artista, gênero)
- `corpus_statistics.json` - Estatísticas do corpus
- `lyrics_corpus.json` - Corpus completo em JSON
- `lyrics_corpus.jsonl` - Corpus em formato JSONL
- `lyrics_corpus.db` - Database SQLite
- `training_corpus.txt` - Corpus em texto puro

**Análises:**
- Padrões convencionais (clichês, estruturas, fórmulas)
- Inovação e efetividade
- Top 20 músicas distintivas

---

**Tempo estimado:** 5-10 minutos

**Requisitos:** 
- Arquivos TXT de letras (disponíveis no repositório ou upload manual)

## 1️⃣ Setup do Ambiente

In [ ]:
# Detectar se está no Google Colab
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

if IN_COLAB:
    print("✓ Google Colab detected")
else:
    print("✓ Running locally")

In [ ]:
# Clone do repositório (ou use pasta existente)
import os

REPO_URL = "https://github.com/guitorte/musicas.git"  # Atualize com o URL correto
WORK_DIR = "/content/musicas" if IN_COLAB else "/home/user/musicas"

if IN_COLAB:
    if not os.path.exists(WORK_DIR):
        print(f"Cloning repository from {REPO_URL}...")
        !git clone $REPO_URL $WORK_DIR
        print("✓ Repository cloned")
    else:
        print("✓ Repository already exists")
        %cd $WORK_DIR
        !git pull
else:
    print(f"✓ Using local directory: {WORK_DIR}")

# Navegar para o diretório
%cd $WORK_DIR
!pwd

In [ ]:
# Instalar dependências
print("Installing dependencies...")

# Dependências Python necessárias
!pip install -q textdistance phonetics unidecode

# Adicionar src ao path
sys.path.insert(0, os.path.join(WORK_DIR, 'lyrics_analysis', 'src'))

print("✓ Dependencies installed")

## 2️⃣ Verificar Arquivos de Letras

Os arquivos TXT com letras devem estar em `/letras/`. Se não existirem, você pode fazer upload manual.

In [ ]:
# Verificar se arquivos de letras existem
import glob

LETRAS_DIR = os.path.join(WORK_DIR, 'letras')
lyrics_files = glob.glob(os.path.join(LETRAS_DIR, '*.txt'))

print(f"Checking for lyrics files in: {LETRAS_DIR}")
print(f"Found {len(lyrics_files)} lyrics files:")

for f in sorted(lyrics_files):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  • {os.path.basename(f)} ({size_mb:.1f} MB)")

if len(lyrics_files) == 0:
    print("\n⚠️  No lyrics files found!")
    print("   The lyrics files are NOT in the git repository (too large).")
    print("   You have two options:")
    print("\n   Option 1: Upload your own TXT files")
    
    if IN_COLAB:
        print("   Run this cell to upload:")
        print("   ```")
        print("   from google.colab import files")
        print("   uploaded = files.upload()")
        print("   # Then move files: !mkdir -p /content/musicas/letras && mv *.txt /content/musicas/letras/")
        print("   ```")
    
    print("\n   Option 2: Use pre-processed corpus files if available")
    print("   (Skip corpus processing and go directly to analysis)")
else:
    print("\n✓ Lyrics files found and ready to process")

In [ ]:
# OPCIONAL: Upload manual de arquivos (descomente se necessário)
# if IN_COLAB:
#     from google.colab import files
#     print("Upload your lyrics TXT files:")
#     uploaded = files.upload()
#     
#     # Move para /letras
#     !mkdir -p $LETRAS_DIR
#     !mv *.txt $LETRAS_DIR/ 2>/dev/null || true
#     print("✓ Files uploaded to /letras")

## 3️⃣ Processar Corpus

Lê os arquivos TXT de `/letras` e gera o corpus estruturado.

**Arquivos gerados:**
- `lyrics_corpus.json` - Corpus completo
- `lyrics_corpus.jsonl` - Formato linha por linha
- `lyrics_corpus.db` - Database SQLite
- `training_corpus.txt` - Texto puro para treinamento
- `corpus_statistics.json` - Estatísticas

In [ ]:
# Processar corpus com caminhos corretos
import subprocess

processor_script = os.path.join(WORK_DIR, 'lyrics_analysis', 'scripts', 'process_corpus.py')
letras_dir = os.path.join(WORK_DIR, 'letras')
output_dir = os.path.join(WORK_DIR, 'lyrics_analysis', 'data', 'processed')

if os.path.exists(processor_script):
    print(f"✓ Found corpus processor: {processor_script}")
    print(f"Processing lyrics from: {letras_dir}")
    print(f"Output to: {output_dir}\n")
    
    # Executar com argumentos explícitos usando subprocess
    result = subprocess.run([
        'python', processor_script,
        '--corpus-path', letras_dir,
        '--output-dir', output_dir
    ])
    
    if result.returncode != 0:
        print(f"\n⚠️  Corpus processing had errors (exit code: {result.returncode})")
    else:
        print("\n✓ Corpus processing complete")
else:
    print("⚠️  Corpus processor not found.")
    print("   Assuming corpus files already exist in data/processed/")
    
    # Listar arquivos existentes
    if os.path.exists(output_dir):
        processed_files = os.listdir(output_dir)
        print(f"\n   Found {len(processed_files)} processed files:")
        for f in sorted(processed_files):
            print(f"     • {f}")

## 4️⃣ Gerar Índice do Corpus

In [ ]:
# Gerar corpus_index.json
index_script = os.path.join(WORK_DIR, 'lyrics_analysis', 'scripts', 'create_corpus_index.py')

if os.path.exists(index_script):
    print("Generating corpus index...")
    !python $index_script
    print("\n✓ Corpus index generated")
else:
    print("⚠️  Index generator not found")

## 5️⃣ Executar Análises

Gera análises de padrões, inovação e distinctiveness.

In [ ]:
# Análise de Padrões Convencionais
pattern_script = os.path.join(WORK_DIR, 'lyrics_analysis', 'scripts', 'analyze_patterns.py')

if os.path.exists(pattern_script):
    print("Running pattern analysis...")
    !python $pattern_script
    print("\n✓ Pattern analysis complete")
else:
    print("⚠️  Pattern analyzer not found")

In [ ]:
# Análise de Inovação e Efetividade
innovation_script = os.path.join(WORK_DIR, 'lyrics_analysis', 'scripts', 'detect_innovations.py')

if os.path.exists(innovation_script):
    print("Running innovation analysis...")
    !python $innovation_script
    print("\n✓ Innovation analysis complete")
else:
    print("⚠️  Innovation analyzer not found")

In [ ]:
# Análise de Distinctiveness (Top 20 músicas distintivas)
distinctive_script = os.path.join(WORK_DIR, 'lyrics_analysis', 'scripts', 'find_distinctive_songs.py')

if os.path.exists(distinctive_script):
    print("Finding distinctive songs...")
    !python $distinctive_script
    print("\n✓ Distinctive songs analysis complete")
else:
    print("⚠️  Distinctive songs finder not found")

## 6️⃣ Verificar Resultados

In [ ]:
# Listar todos os arquivos gerados
import json

print("=" * 80)
print(" ARQUIVOS GERADOS ".center(80, "="))
print("=" * 80)

# Data processada
processed_dir = os.path.join(WORK_DIR, 'lyrics_analysis', 'data', 'processed')
print("\n📁 Data Processada (data/processed/):")
if os.path.exists(processed_dir):
    for f in sorted(os.listdir(processed_dir)):
        if f.startswith('.'):
            continue
        path = os.path.join(processed_dir, f)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✓ {f:<35} {size_mb:>6.1f} MB")

# Análises de padrões
patterns_dir = os.path.join(WORK_DIR, 'lyrics_analysis', 'analysis', 'patterns')
print("\n📊 Análises de Padrões (analysis/patterns/):")
if os.path.exists(patterns_dir):
    for f in sorted(os.listdir(patterns_dir)):
        if f.startswith('.') or not f.endswith('.json'):
            continue
        path = os.path.join(patterns_dir, f)
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✓ {f:<40} {size_kb:>6.1f} KB")

# Análises de inovação
innovations_dir = os.path.join(WORK_DIR, 'lyrics_analysis', 'analysis', 'innovations')
print("\n🚀 Análises de Inovação (analysis/innovations/):")
if os.path.exists(innovations_dir):
    for f in sorted(os.listdir(innovations_dir)):
        if f.startswith('.') or not f.endswith('.json'):
            continue
        path = os.path.join(innovations_dir, f)
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✓ {f:<40} {size_kb:>6.1f} KB")

print("\n" + "=" * 80)
print("\n✅ Pipeline de regeneração concluído!")

## 7️⃣ Estatísticas do Corpus

In [ ]:
# Mostrar estatísticas do corpus
corpus_file = os.path.join(WORK_DIR, 'lyrics_analysis', 'data', 'processed', 'lyrics_corpus.json')

if os.path.exists(corpus_file):
    print("Loading corpus statistics...\n")
    
    with open(corpus_file, 'r', encoding='utf-8') as f:
        corpus = json.load(f)
    
    total_songs = len(corpus['songs'])
    
    # Contar por gênero
    genres = {}
    for song in corpus['songs']:
        genre = song.get('genre', 'Unknown')
        genres[genre] = genres.get(genre, 0) + 1
    
    print("=" * 60)
    print(" CORPUS STATISTICS ".center(60, "="))
    print("=" * 60)
    print(f"\nTotal Songs: {total_songs:,}")
    print("\nGenre Distribution:")
    
    for genre, count in sorted(genres.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / total_songs) * 100
        bar = "█" * int(percentage / 2)
        print(f"  {genre:<12} {count:>4} ({percentage:>5.1f}%) {bar}")
    
    print("\n" + "=" * 60)
else:
    print("⚠️  Corpus file not found")

## 8️⃣ Download dos Resultados (Google Colab)

In [ ]:
# Criar ZIP com todos os resultados para download
if IN_COLAB:
    print("Creating ZIP archive for download...")
    
    !cd $WORK_DIR && zip -r -q lyrics_analysis_results.zip \
        lyrics_analysis/data/processed/*.json \
        lyrics_analysis/data/processed/*.jsonl \
        lyrics_analysis/data/processed/*.db \
        lyrics_analysis/data/processed/*.txt \
        lyrics_analysis/analysis/patterns/*.json \
        lyrics_analysis/analysis/innovations/*.json
    
    print("\n✓ Archive created: lyrics_analysis_results.zip")
    
    # Download
    from google.colab import files
    print("\nDownloading...")
    files.download(os.path.join(WORK_DIR, 'lyrics_analysis_results.zip'))
    print("✓ Download complete")
else:
    print("Not running in Colab - files are already on disk")
    print(f"Location: {WORK_DIR}/lyrics_analysis/")

## 🎉 Pronto!

Todos os arquivos foram regenerados com sucesso.

### Próximos passos:

1. **Explorar análises** - Abra os arquivos JSON em `analysis/` para insights
2. **Usar corpus** - O corpus está disponível em múltiplos formatos em `data/processed/`
3. **Gerar letras** - Use os scripts de geração com os dados processados

### Arquivos principais:

- 📋 `corpus_index.json` - Índice de todas as músicas
- 🎯 `DISTINCTIVE_SONGS.json` - Top 20 músicas mais distintivas
- ⚠️ `AVOID_THESE_CLICHES.json` - Clichês para evitar
- 📊 `innovation_metrics.json` - Métricas de inovação

---

**Dúvidas?** Consulte o README do projeto ou abra uma issue no GitHub.